# Usage of PySpark SQL

In [2]:
!curl -L -O https://raw.githubusercontent.com/mosesyhc/de300-2026wi/refs/heads/main/datasets/pope.txt # grab raw data from github

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 11297  100 11297    0     0  65224      0 --:--:-- --:--:-- --:--:-- 65300


In [3]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                     .appName("Analyzing an unknown article.")
                     .getOrCreate())


In [4]:
## documentation
spark.read??

In [5]:
file_path = r'pope.txt' # fill in

In [6]:
article = spark.read.text(file_path)

In [7]:
article.show(5, truncate=False) # looking at first 5 lines

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                        |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Pope Leo XIV lays out vision of papacy and identifies AI as a main challenge for humanity                                                                                                                                                    |
|By  NICOLE WINFIELD                    

In [8]:
from pyspark.sql.functions import col

In [9]:
# four different ways you can grab the column

article.select(article.value)
article.select(article['value']) # like a dataframe
article.select(col('value'))
article.select('value') # if unambiguous

DataFrame[value: string]

In [12]:
from pyspark.sql.functions import col, split

lines = article.select(
    split(col('value'),
          ' ' # splitting by spaces
          ).alias('line'))

lines.show(5, truncate=False)


+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|line                                                                                                                                                                                                                                                                                  |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[Pope, Leo, XIV, lays, out, vision, of, papacy, and, identifies, AI, as, a, main, challenge, for, humanity]                                                 

In [13]:
lines.printSchema()

root
 |-- line: array (nullable = true)
 |    |-- element: string (containsNull = false)



In [15]:
from pyspark.sql.functions import explode

words = lines.select(explode(col('line')).alias('word'))

words.show(5)

+----+
|word|
+----+
|Pope|
| Leo|
| XIV|
|lays|
| out|
+----+
only showing top 5 rows


In [18]:
words.printSchema()

root
 |-- word: string (nullable = false)



In [22]:
from pyspark.sql.functions import lower

words = words.select(lower(col('word')).alias('word')) # change into lowercase

words.show()


+----------+
|      word|
+----------+
|      pope|
|       leo|
|       xiv|
|      lays|
|       out|
|    vision|
|        of|
|    papacy|
|       and|
|identifies|
|        ai|
|        as|
|         a|
|      main|
| challenge|
|       for|
|  humanity|
|        by|
|          |
|    nicole|
+----------+
only showing top 20 rows


In [25]:
from pyspark.sql.functions import regexp_extract

words_clean = words.select(
    regexp_extract(col('word'), r'(\W+)?([a-z]+)', 0).alias('word')
)

In [24]:
words_clean.show()

+----------+
|      word|
+----------+
|      pope|
|       leo|
|       xiv|
|      lays|
|       out|
|    vision|
|        of|
|    papacy|
|       and|
|identifies|
|        ai|
|        as|
|         a|
|      main|
| challenge|
|       for|
|  humanity|
|        by|
|          |
|    nicole|
+----------+
only showing top 20 rows


In [ ]:
words_nonull = words_clean.filter(  # .where(F.col('word') != ''))

In [26]:
groups = words_clean.groupBy(col('word')) # should use words nonull

In [27]:
groups

GroupedData[grouping expressions: [word], value: [word: string], type: GroupBy]

In [29]:
counts = groups.count()

In [30]:
counts # not actually doing anything yet

DataFrame[word: string, count: bigint]

In [31]:
counts.show(10)

+------------+-----+
|        word|count|
+------------+-----+
|        some|    2|
|   traveling|    2|
|         few|    1|
|    medicare|    1|
|       vocal|    1|
|    received|    1|
|    armangue|    2|
|conversation|    1|
|     explain|    1|
|    hometown|    1|
+------------+-----+
only showing top 10 rows


In [33]:
counts.orderBy(col('count').desc()).show(10)

+----+-----+
|word|count|
+----+-----+
|    |  262|
| the|  137|
|  of|   62|
|  in|   55|
| and|   50|
|  to|   42|
| leo|   36|
|   a|   31|
|pope|   30|
| his|   26|
+----+-----+
only showing top 10 rows


In [35]:
# in classical text analysis, remove filler words, spaces, etc


In [37]:
import pyspark.sql.functions as F # import sql function

counts = ( # there will be sequencing of steps that you need. this goes into designing a pipeline
    spark.read.text(file_path)
    .select(F.split(F.col('value'), ' ').alias('line'))
    .select(F.explode(F.col('line')).alias('word'))
    .select(F.lower(F.col('word')).alias('word'))
    .select(F.regexp_extract(F.col('word'), r'(\W+)?([a-z]+)', 2).alias('word'))
    .where(F.col('word') != '')
    .groupBy(F.col('word'))
    .count()
    )


In [40]:
# trying to count number of word with certain lengths

counts_by_length = (
    spark.read.text(file_path)
    .select(F.split(F.col('value'), ' ').alias('line'))
    .select(F.explode(F.col('line')).alias('word'))
    .select(F.lower(F.col('word')).alias('word'))
    .select(F.regexp_extract(F.col('word'), r'(\W+)?([a-z]+)', 2).alias('word'))
    .where(F.col('word') != '')
    .groupBy(F.length(F.col('word'))) # grouping by length of word
    #.groupBy(F.col('word')) # grouping by word
    .count()
    )

In [43]:
counts_by_length.orderBy(F.col('count')).show()

+------------+-----+
|length(word)|count|
+------------+-----+
|          15|    1|
|          14|    3|
|          13|    8|
|          12|   16|
|          11|   26|
|           1|   35|
|          10|   62|
|           9|   82|
|           6|  118|
|           8|  139|
|           5|  157|
|           7|  167|
|           4|  238|
|           2|  313|
|           3|  392|
+------------+-----+



In [49]:
letters = (
    spark.read.text(file_path)
    .select(F.split(F.col('value'), ' ').alias('line'))
    .select(F.explode(F.col('line')).alias('word'))
    .select(F.lower(F.col('word')).alias('word'))
    .select(F.regexp_extract(F.col('word'), r'(\W+)?([a-z]+)', 2).alias('word'))
    .where(F.col('word') != '')
    .select(F.explode(F.split(F.col('word'), '')).alias('letter'))
    .groupBy(F.col('letter'))
    .count()
    )

In [52]:
letters.orderBy(F.col('letter')).show(26)

+------+-----+
|letter|count|
+------+-----+
|     a|  807|
|     b|   72|
|     c|  355|
|     d|  338|
|     e| 1005|
|     f|  208|
|     g|  157|
|     h|  388|
|     i|  706|
|     j|   10|
|     k|   32|
|     l|  364|
|     m|  215|
|     n|  619|
|     o|  657|
|     p|  259|
|     q|    3|
|     r|  489|
|     s|  527|
|     t|  741|
|     u|  188|
|     v|  141|
|     w|   97|
|     x|   30|
|     y|  137|
|     z|   15|
+------+-----+

